## apptrain et apptest

In [1]:
import pandas as pd
import numpy as np
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Charger les données brutes
application_train = pd.read_csv('../csv_files/application_train.csv')
application_test = pd.read_csv('../csv_files/application_test.csv')

# Séparer les features et la target pour le dataset d'entraînement
X_train_raw = application_train.drop(columns=['TARGET'])
y_train_raw = application_train['TARGET']

# Utiliser les données de test comme données de production (sans la target)
X_test_raw = application_test

# Identifier les colonnes numériques et non numériques
numeric_features = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train_raw.select_dtypes(exclude=[np.number]).columns.tolist()

# Créer les pipelines de prétraitement pour les données numériques et catégorielles
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combiner les transformations en un préprocesseur
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Appliquer les transformations aux données d'entraînement et de test
X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

# Convertir les numpy arrays en DataFrames
X_train_resampled = pd.DataFrame(X_train_processed, columns=preprocessor.get_feature_names_out())
X_test_resampled = pd.DataFrame(X_test_processed, columns=preprocessor.get_feature_names_out())

# Créer un rapport Evidently pour le data drift avec des métriques spécifiques
data_drift_report = Report(
    metrics=[
        DataDriftPreset(),
    ]
)

# Générer le rapport
data_drift_report.run(reference_data=X_train_resampled, current_data=X_test_resampled)

# Sauvegarder le rapport dans un fichier HTML
data_drift_report.save_html('data_drift_report.html')

print("Data drift report saved to 'data_drift_report.html'")


/home/alexandre/anaconda3/envs/stabadenvP7/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/alexandre/anaconda3/envs/stabadenvP7/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/alexandre/anaconda3/envs/stabadenvP7/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/alexandre/anaconda3/envs/stabadenvP7/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Data drift report saved to 'data_drift_report.html'
